In [1]:
# Install Inkscape and libdmtx (required for pylibdmtx)
!sudo apt-get update
!sudo apt-get install -y libdmtx0b
!sudo apt-get install -y inkscape

# Install Python dependencies
!pip install pylibdmtx Pillow
!pip install reportlab PyPDF2 Pillow


Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Get:8 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,637 kB]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,619 kB]
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,869 kB]
Get:14 https://ppa.lau

In [2]:
import os
from pylibdmtx.pylibdmtx import encode
from PIL import Image

def generate_datamatrix_headless(base_data, start, end, output_folder):
    os.makedirs(output_folder, exist_ok=True)
    field_21_value = base_data.split('(21)')[-1].strip()

    for i in range(start, end + 1):
        data = base_data + f'{i:04d}'
        encoded = encode(data.encode('utf8'))
        img = Image.frombytes('RGB', (encoded.width, encoded.height), encoded.pixels)

        # Scale 4x
        img = img.resize((encoded.width * 4, encoded.height * 4), Image.Resampling.NEAREST)

        filename = f'{field_21_value}{i:04d}.png'
        img.save(os.path.join(output_folder, filename), dpi=(600, 600))
        if i % 100 == 0: print(f"Generated {i}...")

# Usage
BASE_DATA = "(01)8906172600074\n(11)260115\n(10)2601003\n(21)2601TURHAN14"
generate_datamatrix_headless(BASE_DATA, 1, 3, "/content/dm_codes")

In [5]:
import os
import subprocess
import xml.etree.ElementTree as ET
import base64
import concurrent.futures
import threading
import sys

def process_single_label(
    index,
    serial_number,
    svg_template_path,
    datamatrix_folder,
    output_folder,
    image_id,
    secondary_barcode_folder,
    secondary_image_id,
    transparent_bg
):
    # Create a unique temp file for this thread to avoid collisions
    thread_id = threading.get_ident()
    temp_svg = f"temp_{serial_number}_{thread_id}.svg"

    try:
        # 1. Parse the SVG Template
        tree = ET.parse(svg_template_path)
        root = tree.getroot()

        # Registers namespaces to prevent "ns0:" prefixes in output
        # (This helps keep the SVG clean for Inkscape)
        for _, node in ET.iterparse(svg_template_path, events=['start-ns']):
            if node[0]:
                ET.register_namespace(node[0], node[1])

        # 2. Replace Serial Number Text
        # Looks for any text element containing "serial_number_text"
        for elem in root.iter():
            if elem.text and "serial_number_text" in elem.text:
                elem.text = elem.text.replace("serial_number_text", serial_number)

        # 3. Embed Primary DataMatrix Image (Base64)
        # We assume the DM filename matches the serial number (e.g., "PREFIX0001.png")
        dm_image_path = os.path.join(datamatrix_folder, f"{serial_number}.png")

        if os.path.exists(dm_image_path):
            with open(dm_image_path, "rb") as img_file:
                img_base64 = base64.b64encode(img_file.read()).decode("utf-8")
                href = f"data:image/png;base64,{img_base64}"

                # Find the image element by ID and update href
                found_img = False
                for img in root.iter():
                    if img.attrib.get("id") == image_id:
                        # Handle xlink namespace commonly used in SVGs
                        key = "{http://www.w3.org/1999/xlink}href"
                        if key in img.attrib:
                            img.attrib[key] = href
                        else:
                            # Fallback for standard href
                            img.attrib["href"] = href
                        found_img = True
                        break
                if not found_img:
                    print(f"Warning: Image ID '{image_id}' not found in SVG for {serial_number}")
        else:
            print(f"Error: DataMatrix file missing: {dm_image_path}")
            return

        # 4. Embed Secondary Barcode (Optional)
        if secondary_barcode_folder and secondary_image_id:
            sec_path = os.path.join(secondary_barcode_folder, f"{serial_number}.png")
            if os.path.exists(sec_path):
                with open(sec_path, "rb") as f:
                    b64 = base64.b64encode(f.read()).decode("utf-8")
                    href = f"data:image/png;base64,{b64}"
                    for img in root.iter():
                        if img.attrib.get("id") == secondary_image_id:
                            key = "{http://www.w3.org/1999/xlink}href"
                            if key in img.attrib:
                                img.attrib[key] = href
                            else:
                                img.attrib["href"] = href
                            break

        # 5. Save Modified Temp SVG
        tree.write(temp_svg)

        # 6. Run Inkscape CLI to export PNG
        output_filename = os.path.join(output_folder, f"label_{serial_number}.png")

        cmd = [
            "inkscape",
            temp_svg,
            "--export-type=png",
            "--export-area-page",
            "--export-dpi=600",
            f"--export-filename={output_filename}"
        ]

        if not transparent_bg:
            cmd.extend(["--export-background=white", "--export-background-opacity=255"])

        subprocess.run(cmd, check=True, capture_output=True)

    except subprocess.CalledProcessError as e:
        print(f"Inkscape Error for {serial_number}: {e.stderr.decode()}")
    except Exception as e:
        print(f"Python Error for {serial_number}: {str(e)}")
    finally:
        # Cleanup temp file
        if os.path.exists(temp_svg):
            os.remove(temp_svg)

def generate_labels_headless(config):
    # Verify paths
    if not os.path.exists(config['svg_path']):
        print(f"Error: Template not found at {config['svg_path']}")
        return
    if not os.path.exists(config['datamatrix_folder']):
        print(f"Error: DataMatrix folder not found at {config['datamatrix_folder']}")
        return

    os.makedirs(config['output_folder'], exist_ok=True)

    tasks = []
    # Use CPU count to determine workers (Colab usually has 2-4 vCPUs)
    max_workers = os.cpu_count() or 4

    print(f"Starting generation with {max_workers} workers...")

    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        for i in range(config['start_number'], config['end_number'] + 1):
            serial_number = f"{config['serial_prefix']}{i:04d}"

            tasks.append(executor.submit(
                process_single_label,
                i,
                serial_number,
                config['svg_path'],
                config['datamatrix_folder'],
                config['output_folder'],
                config['image_id'],
                config.get('secondary_folder'),
                config.get('secondary_image_id'),
                config.get('transparent_bg', True)
            ))

        # Progress Monitor
        total = len(tasks)
        completed = 0
        for future in concurrent.futures.as_completed(tasks):
            completed += 1
            if completed % 10 == 0 or completed == total:
                print(f"Progress: {completed}/{total} labels generated")

    print(f"Done! Check folder: {config['output_folder']}")

# ==========================================
# CONFIGURATION
# ==========================================
# Update these paths based on your Colab files
config = {
    "svg_path": "/content/TUR-HAN-14-MDF-12-UDI LABEL-V2.svg",       # Upload your .svg file to Colab
    "datamatrix_folder": "/content/dm_codes",       # Folder containing your generated DM codes
    "output_folder": "/content/final_labels",       # Where results will go

    "start_number": 1,                              # Start index
    "end_number": 3,                                # End index (based on your screenshot)

    # CRITICAL: This must combine with the number to match your DM filenames
    # Example: If file is "2601TURHAN140001.png", prefix is "2601TURHAN14"
    "serial_prefix": "2601TURHAN14",

    "image_id": "image1",                           # The ID of the image in your SVG to replace
    "transparent_bg": False,

    # Optional Secondary Barcode
    "secondary_folder": None,
    "secondary_image_id": None
}

# Run it
generate_labels_headless(config)


Starting generation with 2 workers...
Progress: 3/3 labels generated
Done! Check folder: /content/final_labels


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import shutil
import os

folder_to_delete = '/content/dm_codes'

if os.path.exists(folder_to_delete):
    shutil.rmtree(folder_to_delete)
    print(f"Folder '{folder_to_delete}' and its contents deleted successfully.")
else:
    print(f"Folder '{folder_to_delete}' does not exist.")

Folder '/content/dm_codes' and its contents deleted successfully.


In [6]:
import os
import re
import sys

def verify_labels_headless(folder_path):
    print(f"Checking folder: {folder_path}")
    print("=" * 40)

    if not os.path.exists(folder_path):
        print(f"Error: Folder '{folder_path}' does not exist.")
        return

    # 1. Gather all files
    all_files = [f for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f))]

    if not all_files:
        print("❌ No files found in the selected folder.")
        return

    # 2. Extract Numbers
    last_four_digits = []
    invalid_files = []

    for filename in all_files:
        name_without_ext = os.path.splitext(filename)[0]

        # Look for 4 digits at the end
        match = re.search(r'(\d{4})$', name_without_ext)
        if match:
            last_four_digits.append(int(match.group(1)))
        else:
            # Fallback: try any digits at end
            match = re.search(r'(\d+)$', name_without_ext)
            if match and len(match.group(1)) >= 4:
                last_four_digits.append(int(match.group(1)[-4:]))
            else:
                invalid_files.append(filename)

    print(f"Total valid files processed: {len(last_four_digits)}")

    if invalid_files:
        print("\n⚠️  Ignored files (no 4-digit suffix):")
        for f in invalid_files:
            print(f"   • {f}")

    if not last_four_digits:
        print("\n❌ No valid numbered files found.")
        return

    # 3. Check Sequence
    sorted_numbers = sorted(last_four_digits)

    # Check Duplicates
    seen = set()
    duplicates = set()
    for num in sorted_numbers:
        if num in seen:
            duplicates.add(num)
        seen.add(num)

    # Check Missing Numbers
    missing_numbers = []
    if sorted_numbers:
        expected = sorted_numbers[0]
        for current in sorted_numbers:
            while expected < current:
                missing_numbers.append(expected)
                expected += 1
            expected = current + 1

    # 4. Report Results
    print("\nRESULTS")
    print("=" * 25)

    issues_found = False

    if duplicates:
        issues_found = True
        print("❌ DUPLICATES FOUND:")
        for num in sorted(list(duplicates)):
            print(f"   • {num:04d}")

    if missing_numbers:
        issues_found = True
        print("❌ MISSING NUMBERS:")
        for num in missing_numbers:
            print(f"   • {num:04d}")

    if not issues_found:
        print("✅ VERIFICATION SUCCESSFUL!")
        print("Perfect arithmetic progression with increment of 1.")
        print(f"Sequence: {sorted_numbers[0]:04d} to {sorted_numbers[-1]:04d}")

# ==========================================
# USAGE
# ==========================================
FOLDER_TO_CHECK = "/content/final_labels"  # Update this to your output folder
verify_labels_headless(FOLDER_TO_CHECK)


Checking folder: /content/final_labels
Total valid files processed: 3

RESULTS
✅ VERIFICATION SUCCESSFUL!
Perfect arithmetic progression with increment of 1.
Sequence: 0001 to 0003


In [10]:
import os
import time
from PIL import Image
import PyPDF2
from reportlab.pdfgen import canvas

def convert_png_to_pdf_headless(input_folder, output_filename, use_image_size=True, custom_width_mm=210, custom_height_mm=297):
    if not os.path.exists(input_folder):
        print(f"Error: Folder '{input_folder}' not found.")
        return

    # 1. Get and Sort Files
    files = sorted([
        os.path.join(input_folder, f)
        for f in os.listdir(input_folder)
        if f.lower().endswith('.png')
    ])

    if not files:
        print("No PNG files found.")
        return

    print(f"Found {len(files)} images.")

    # 2. Determine Page Size
    if use_image_size:
        try:
            first_img = Image.open(files[0])
            first_size = first_img.size

            # Get DPI (Default to 300 if not found, same as your GUI script)
            dpi = 300
            if hasattr(first_img, 'info') and 'dpi' in first_img.info:
                # Handle tuple vs single value
                dpi = first_img.info['dpi'][0] if isinstance(first_img.info['dpi'], tuple) else first_img.info['dpi']

            print(f"Detected Image Properties - Size: {first_size}px, DPI: {dpi}")

            # Calculate physical size in millimeters
            # Logic matches your original: (pixels / dpi) * 25.4
            width_mm = (first_size[0] / dpi) * 25.4
            height_mm = (first_size[1] / dpi) * 25.4

            print(f"Setting PDF Page Size to: {width_mm:.2f}mm x {height_mm:.2f}mm")

        except Exception as e:
            print(f"Error detecting image size: {e}")
            return
    else:
        width_mm = custom_width_mm
        height_mm = custom_height_mm
        print(f"Using Custom Page Size: {width_mm}mm x {height_mm}mm")

    # Convert mm to points for ReportLab (1 mm = 2.83465 points)
    PAGE_WIDTH = width_mm * 2.83465
    PAGE_HEIGHT = height_mm * 2.83465
    PAGE_SIZE = (PAGE_WIDTH, PAGE_HEIGHT)

    # 3. Process Images
    temp_pdfs = []
    try:
        for i, img_path in enumerate(files):
            temp_pdf = f"temp_page_{i}.pdf"
            temp_pdfs.append(temp_pdf)

            img = Image.open(img_path)

            # Create PDF Page
            c = canvas.Canvas(temp_pdf, pagesize=PAGE_SIZE)

            # If using image size, we draw exact 1:1 fit
            if use_image_size:
                c.drawImage(img_path, 0, 0, width=PAGE_WIDTH, height=PAGE_HEIGHT)
            else:
                # If using custom size (e.g. A4), verify scaling logic
                # Calculate scaling to fit the page while maintaining aspect ratio
                img_ratio = img.width / img.height
                page_ratio = PAGE_WIDTH / PAGE_HEIGHT

                if img_ratio > page_ratio:
                    draw_width = PAGE_WIDTH
                    draw_height = draw_width / img_ratio
                else:
                    draw_height = PAGE_HEIGHT
                    draw_width = draw_height * img_ratio

                # Center the image
                x_pos = (PAGE_WIDTH - draw_width) / 2
                y_pos = (PAGE_HEIGHT - draw_height) / 2
                c.drawImage(img_path, x_pos, y_pos, width=draw_width, height=draw_height)

            c.showPage()
            c.save()

            if (i + 1) % 10 == 0:
                print(f"Processed {i + 1}/{len(files)}...")

        # 4. Merge
        print("Merging pages into final PDF...")
        merger = PyPDF2.PdfMerger()
        for pdf in temp_pdfs:
            merger.append(pdf)

        merger.write(output_filename)
        merger.close()

        print(f"✅ Success! Saved to: {output_filename}")

    except Exception as e:
        print(f"Error during conversion: {e}")
    finally:
        # Cleanup
        for pdf in temp_pdfs:
            if os.path.exists(pdf):
                os.remove(pdf)

# ==========================================
# USAGE
# ==========================================
INPUT_FOLDER = "/content/final_labels"
OUTPUT_FILE = "/content/final_labels_merged.pdf"

# Set use_image_size=True to automatically match the PNG dimensions
convert_png_to_pdf_headless(
    INPUT_FOLDER,
    OUTPUT_FILE,
    use_image_size=True
)


Found 3 images.
Detected Image Properties - Size: (2362, 945)px, DPI: 599.9988
Setting PDF Page Size to: 99.99mm x 40.01mm
Merging pages into final PDF...
✅ Success! Saved to: /content/final_labels_merged.pdf
